In [7]:
import os
import mne
import numpy as np
from datetime import datetime

def calculate_epoch_durations(directory_path, output_file="epoch_durations.txt", fs=256):
    """
    Calculate duration of all .fif epoch files in a directory and save to text file.
    
    Parameters:
        directory_path: str, path to directory containing .fif files
        output_file: str, name of output text file
        fs: int, sampling frequency (default 256)
    """
    
    # Get all .fif files in directory
    fifFiles = [f for f in os.listdir(directory_path) if f.endswith('.fif')]
    
    if not fifFiles:
        print("No .fif files found in the directory.")
        return
    
    totalDuration = 0
    results = []
    
    print(f"Processing {len(fifFiles)} .fif epoch files...")
    
    for i, filename in enumerate(fifFiles):
        try:
            filepath = os.path.join(directory_path, filename)
            
            # Read the .fif file as epochs (suppress output)
            with mne.utils.use_log_level('ERROR'):
                epochs = mne.read_epochs(filepath, preload=False)
            
            # Calculate duration in minutes
            nEpochs = len(epochs)
            nSamplesPerEpoch = epochs.get_data().shape[-1]  # samples per epoch
            durationMinutes = (nEpochs * nSamplesPerEpoch) / (fs * 60)
            
            totalDuration += durationMinutes
            results.append((filename, durationMinutes, nEpochs))
            
            print(f"Processed {i+1}/{len(fifFiles)}: {filename} - {nEpochs} epochs, {durationMinutes:.2f} min")
            
        except Exception as e:
            print(f"Error processing {filename}: {str(e)}")
            results.append((filename, f"Error: {str(e)}", 0))
    
    # Write results to text file
    with open(output_file, 'w') as f:
        f.write("FIF Epoch Files Duration Report\n")
        f.write("=" * 60 + "\n")
        f.write(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Directory: {directory_path}\n")
        f.write(f"Sampling Rate: {fs} Hz\n")
        f.write(f"Total Files: {len(fifFiles)}\n")
        f.write("=" * 60 + "\n\n")
        
        f.write("File Name\t\tN Epochs\tDuration (minutes)\n")
        f.write("-" * 60 + "\n")
        
        for result in results:
            if len(result) == 3:
                filename, duration, nEpochs = result
                if isinstance(duration, float):
                    f.write(f"{filename}\t\t{nEpochs}\t\t{duration:.2f}\n")
                else:
                    f.write(f"{filename}\t\t{nEpochs}\t\t{duration}\n")
        
        f.write("-" * 60 + "\n")
        f.write(f"Total Duration: {totalDuration:.2f} minutes ({totalDuration/60:.2f} hours)\n")
    
    print(f"\nResults saved to: {output_file}")
    print(f"Total duration: {totalDuration:.2f} minutes ({totalDuration/60:.2f} hours)")

directoryPath = "/Users/wachiii/Workschii/brain-asd/data/data_adults_eyeclose/epoch30sFiles/asd"  # Change this path
calculate_epoch_durations(directoryPath, "EC_ASD_Adult_durations.txt", fs=256)

Processing 35 .fif epoch files...
Loading data for 6 events and 7680 original time points ...
Processed 1/35: A28_EC.fif - 6 epochs, 3.00 min
Loading data for 6 events and 7680 original time points ...
Processed 2/35: A24_EC.fif - 6 epochs, 3.00 min
Loading data for 6 events and 7680 original time points ...
Processed 3/35: A12_EC.fif - 6 epochs, 3.00 min
Loading data for 6 events and 7680 original time points ...
Processed 4/35: A26_EC.fif - 6 epochs, 3.00 min
Loading data for 6 events and 7680 original time points ...
Processed 5/35: A34_EC.fif - 6 epochs, 3.00 min
Loading data for 6 events and 7680 original time points ...
Processed 6/35: A02_EC.fif - 6 epochs, 3.00 min
Loading data for 6 events and 7680 original time points ...
Processed 7/35: A10_EC.fif - 6 epochs, 3.00 min
Loading data for 6 events and 7680 original time points ...
Processed 8/35: A14_EC.fif - 6 epochs, 3.00 min
Loading data for 6 events and 7680 original time points ...
Processed 9/35: A06_EC.fif - 6 epochs, 3.0

In [ ]:
import os
import mne
import numpy as np
from datetime import datetime

def calculate_fif_durations_robust(directory_path, output_file="fif_durations.txt", fs=256):
    """
    Calculate duration of all .fif files trying multiple reading methods.
    
    Parameters:
        directory_path: str, path to directory containing .fif files
        output_file: str, name of output text file
        fs: int, sampling frequency (default 256)
    """
    
    # Get all .fif files in directory
    fifFiles = [f for f in os.listdir(directory_path) if f.endswith('.fif')]
    
    if not fifFiles:
        print("No .fif files found in the directory.")
        return
    
    totalDuration = 0
    results = []
    
    print(f"Processing {len(fifFiles)} .fif files...")
    
    for i, filename in enumerate(fifFiles):
        try:
            filepath = os.path.join(directory_path, filename)
            durationMinutes = None
            nEpochs = None
            method_used = ""
            
            # Method 1: Try reading as epochs
            try:
                with mne.utils.use_log_level('ERROR'):
                    epochs = mne.read_epochs(filepath, preload=False)
                nEpochs = len(epochs)
                nSamplesPerEpoch = epochs.get_data(picks=[0]).shape[-1] if len(epochs) > 0 else 0
                durationMinutes = (nEpochs * nSamplesPerEpoch) / (fs * 60)
                method_used = "epochs"
            except:
                pass
            
            # Method 2: Try reading as raw data
            if durationMinutes is None:
                try:
                    with mne.utils.use_log_level('ERROR'):
                        raw = mne.io.read_raw_fif(filepath, preload=False)
                    nSamples = raw.n_times
                    durationMinutes = nSamples / (fs * 60)
                    method_used = "raw"
                except:
                    pass
            
            # Method 3: Try reading as evoked data
            if durationMinutes is None:
                try:
                    with mne.utils.use_log_level('ERROR'):
                        evoked = mne.read_evokeds(filepath)
                    if isinstance(evoked, list):
                        evoked = evoked[0]
                    nSamples = len(evoked.times)
                    durationMinutes = nSamples / (fs * 60)
                    method_used = "evoked"
                except:
                    pass
            
            # Method 4: Try generic fiff reading to get basic info
            if durationMinutes is None:
                try:
                    with mne.utils.use_log_level('ERROR'):
                        fiff_info = mne.io.read_info(filepath)
                    # This might not give us duration, but at least we know it's readable
                    durationMinutes = "Unknown - readable file"
                    method_used = "info_only"
                except:
                    pass
            
            if durationMinutes is not None:
                if isinstance(durationMinutes, float):
                    totalDuration += durationMinutes
                    if nEpochs:
                        results.append((filename, durationMinutes, nEpochs, method_used))
                        print(f"Processed {i+1}/{len(fifFiles)}: {filename} - {nEpochs} epochs, {durationMinutes:.2f} min ({method_used})")
                    else:
                        results.append((filename, durationMinutes, "N/A", method_used))
                        print(f"Processed {i+1}/{len(fifFiles)}: {filename} - {durationMinutes:.2f} min ({method_used})")
                else:
                    results.append((filename, durationMinutes, "N/A", method_used))
                    print(f"Processed {i+1}/{len(fifFiles)}: {filename} - {durationMinutes}")
            else:
                error_msg = f"Unable to read with any method"
                results.append((filename, f"Error: {error_msg}", "N/A", "failed"))
                print(f"Error processing {filename}: {error_msg}")
            
        except Exception as e:
            print(f"Error processing {filename}: {str(e)}")
            results.append((filename, f"Error: {str(e)}", "N/A", "failed"))
    
    # Write results to text file
    with open(output_file, 'w') as f:
        f.write("FIF Files Duration Report (Robust)\n")
        f.write("=" * 70 + "\n")
        f.write(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Directory: {directory_path}\n")
        f.write(f"Sampling Rate: {fs} Hz\n")
        f.write(f"Total Files: {len(fifFiles)}\n")
        f.write("=" * 70 + "\n\n")
        
        f.write("File Name\t\tN Epochs\tDuration (min)\tMethod\n")
        f.write("-" * 70 + "\n")
        
        for result in results:
            filename, duration, epochs, method = result
            if isinstance(duration, float):
                f.write(f"{filename}\t\t{epochs}\t\t{duration:.2f}\t\t{method}\n")
            else:
                f.write(f"{filename}\t\t{epochs}\t\t{duration}\t{method}\n")
        
        f.write("-" * 70 + "\n")
        f.write(f"Total Duration: {totalDuration:.2f} minutes ({totalDuration/60:.2f} hours)\n")
    
    print(f"\nResults saved to: {output_file}")
    print(f"Total duration: {totalDuration:.2f} minutes ({totalDuration/60:.2f} hours)")


directoryPath = "/Users/wachiii/Workschii/brain-asd/data/data_adults_eyeclose/epoch30sFiles/asd"  # Change this path
calculate_fif_durations_robust(directoryPath, "EO_ASD_Adult_durations.txt", fs=256)